<img src = "https://drive.google.com/uc?export=view&id=1Hh_G3M13P9xSNgSiQ-WnALg93XwK_hG8" alt = "Encabezado MLDS" width = "100%">  </img>

# **Tarea 5 - *Modelos generativos***
---

En esta tarea utilizarás Transformers para convertir texto natural en representaciones mediante emojis. El ejercicio permite comprender cómo los modelos generativos capturan la semántica del texto y su contexto emocional.

> **Importante: Recomendamos utilizar GPU para la ejecución de este notebook, ya que puede tomar mucho tiempo la ejeución de algunos casos de prueba en caso de que no se utilice. Tenga en cuenta que, con el propósito de guardar recursos, los primeros puntos se pueden solucionar sin utilizarlo**

**Descripción del problema**
---
La representación de texto mediante emojis trasciende la traducción directa palabra-a-símbolo. Requiere que el modelo:
- Comprenda el contexto semántico del texto de entrada
- Identifique conceptos clave y emociones implícitas
- Genere secuencias coherentes de emojis que preserven el significado original
- Mantenga la fluidez en la representación visual

# 📘 Tarea 5 · Versión explicada

Copia del notebook resuelto con una explicación antes de cada celda de código. El código y las salidas son los de la ejecución real que obtuvo **100 % en UNCode**, con las 11 pruebas en 100. El original sin explicaciones es `v2_M5U5_Tarea_5_Resuelto.ipynb`.

**Cómo leer cada explicación**
- **📘 Objetivo de la celda:** para qué existe la celda dentro del flujo.
- **Qué hace cada parte del código:** cada instrucción relevante y su función.
- **Punto clave:** el detalle que conviene recordar, cuando lo hay.
- **🔎 Resultado:** la lectura de la salida obtenida.

| | |
|---|---|
| **Problema** | Convertir frases en inglés en secuencias de emojis: secuencia a secuencia, como la traducción automática |
| **Modelo** | Transformer codificador-decodificador. El codificador lee el inglés; el decodificador escribe los emojis uno a uno y consulta al codificador con atención cruzada |
| **Datos** | Unas 10.000 parejas de frase y emojis: 6.605 de entrenamiento y 3.129 de prueba |
| **Resultado** | La pérdida bajó de 8,03 a 3,48; la mejor validación fue 3,11 en la época 19 |

```
JSON frase → emojis
   → Punto 1 preprocess      limpieza y [BOS] … [EOS]
   → Punto 2 vectorize       enteros de longitud 26
   → Punto 3 codificador     (lote, 26)            → (lote, 26, 128)
   → Punto 4 decodificador   (lote, 25) + contexto → (lote, 25, 5000)
   → Punto 5 get_model       une ambas capas
   → Punto 6 train_model     entrada [:, :-1], objetivo [:, 1:], 40 épocas
   → Generación token a token: [BOS] → … → [EOS]
```

### 📘 Instalación del calificador

**Objetivo de la celda.** Preparar el entorno para que el notebook pueda enviarse a UNCode.

**Qué hace cada parte del código**

| Parte del código | Qué hace |
|---|---|
| `%%capture` | Oculta la salida de la celda |
| `!pip install rlxcrypt` | Instala la librería que descifra el módulo del calificador |
| `!wget ... session.pye` | Descarga el módulo cifrado del calificador |

In [39]:
%%capture
!pip install rlxcrypt
!wget --no-cache -O session.pye -q https://raw.githubusercontent.com/JuezUN/INGInious/master/external%20libs/session.pye

### 📘 Inicio de sesión en UNCode

**Objetivo de la celda.** Crear el objeto `grader`, que ejecuta las pruebas y registra la entrega de esta tarea.

**Qué hace cada parte del código**

| Parte del código | Qué hace |
|---|---|
| `import rlxcrypt, session` | Carga el descifrador y el módulo del calificador |
| `session.LoginSequence("DL-GroupMLDS-5-2026-2@...")` | Abre la sesión del curso y de la tarea; pide usuario y contraseña |

> **Punto clave.** Todas las celdas `grader.` dependen de esta. La contraseña no queda guardada.

In [40]:
import rlxcrypt
import session

grader = session.LoginSequence("DL-GroupMLDS-5-2026-2@8c487bd7-4c95-4b75-88d5-9e7093aae729")

Please enter your UNCode username: nremolina
Please enter your password: ··········


> Al finalizar, podrás revisar en UNCode tus submissions en la [página de la tarea.](https://uncode.unal.edu.co/course/DL-GroupMLDS-5-2026-2/8c487bd7-4c95-4b75-88d5-9e7093aae729)

### 📘 Librerías y semillas

**Objetivo de la celda.** Importar las herramientas del notebook y fijar el azar para que los resultados sean reproducibles.

**Qué hace cada parte del código**

| Parte del código | Qué hace |
|---|---|
| `import keras` | Construcción y entrenamiento de la red |
| `import numpy as np, pandas as pd` | Arreglos numéricos y tablas |
| `import re` | Expresiones regulares para limpiar el texto |
| `import json` | Lectura del archivo de datos |
| `random.seed(0), keras.utils.set_random_seed(0), np.random.seed(0)` | Fijan el azar de Python, Keras y NumPy |

> **Punto clave.** El calificador compara valores numéricos exactos: sin semillas, los pesos iniciales nunca coincidirían con los de la referencia.

In [41]:
import keras
import numpy as np
import matplotlib.pyplot as plt
import os, random
import pandas as pd
import re
import json


%matplotlib inline
plt.style.use("ggplot")
# Seleccionamos las semillas para efectos de reproducibilidad
random.seed(0)
keras.utils.set_random_seed(0)
np.random.seed(0)

### 📘 Versiones del entorno

**Objetivo de la celda.** Registrar con qué versiones se ejecutó el notebook.

**Qué hace cada parte del código**

| Parte del código | Qué hace |
|---|---|
| `!python --version` | Muestra la versión de Python |
| `np.__version__, keras.__version__` | Muestran las versiones de NumPy y Keras |

In [42]:
# Versiones de las librerías usadas.
!python --version
print('NumPy', np.__version__)
print('keras', keras.__version__)

Python 3.13.16
NumPy 2.4.6
keras 3.13.2


#### 🔎 Resultado

Versiones más nuevas que las del enunciado, que usó Python 3.11.12, NumPy 2.0.2 y Keras 3.8.0. Todas las pruebas pasaron igual.

Esta actividad se realizó con las siguientes versiones:
* Python 3.11.12
* NumPy 2.0.2
* keras 3.8.0

## **Cargar Datos**
---

Vamos cargar nuestro conjunto de datos. Este posee dos columnas: la primera contiene 10,000 textos en inglés  y en la segunda su representación por medio de emojis. Si bien esta puede parecer una tarea de traducción simple, el uso de emojis como concepto le exige al modelo la capacidad de abstraer patrones complejos.

### 📘 Descarga de los datos

**Objetivo de la celda.** Traer el archivo de parejas frase y emojis al entorno de Colab.

**Qué hace cada parte del código**

| Parte del código | Qué hace |
|---|---|
| `!pip install gdown` | Instala la herramienta que descarga archivos de Google Drive |
| `!gdown https://drive.google.com/uc?id=...` | Descarga `emoji_translations.json` a partir de su identificador |

In [43]:
!pip install gdown
!gdown https://drive.google.com/uc?id=1HjGoRLLnLdEWjB1peXGFQXNGvbPRaAlj

Downloading...
From: https://drive.google.com/uc?id=1HjGoRLLnLdEWjB1peXGFQXNGvbPRaAlj
To: /content/emoji_translations.json
100% 1.09M/1.09M [00:00<00:00, 12.0MB/s]


### 📘 Lectura y división en entrenamiento y prueba

**Objetivo de la celda.** Convertir el archivo en una tabla de dos columnas y separarla en entrenamiento y prueba.

**Qué hace cada parte del código**

| Parte del código | Qué hace |
|---|---|
| `open(..., encoding='utf-8')` | Abre el archivo con la codificación que admite emojis |
| `json.load(f)` | Lo convierte en un diccionario `{frase: emojis}` |
| `pd.DataFrame(data_dict.items(), columns=['ingles', 'emojis'])` | Crea una fila por pareja con las columnas `ingles` y `emojis` |
| `data.dropna()` | Quita las filas vacías; no renumera el índice |
| `data.loc[:int(len(data)*0.7)]` | Toma hasta la etiqueta del 70 % como entrenamiento, incluida esa fila |
| `data.loc[int(len(data)*0.7):]` | Toma desde esa etiqueta hasta el final como prueba |

> **Punto clave.** Como `.loc` incluye el extremo y `dropna()` dejó huecos en el índice, la división real es 67,9 % y 32,1 %, y la fila de la frontera queda en ambos conjuntos. Se dejó así porque viene en la plantilla y el calificador espera 6.605 filas de entrenamiento.

In [44]:
with open("/content/emoji_translations.json", 'r', encoding='utf-8') as f:
    data_dict = json.load(f)

data = pd.DataFrame(data_dict.items(), columns=['ingles', 'emojis'])
data = data.dropna()

train_texts = data.loc[:int(len(data)*0.7)]
test_texts = data.loc[int(len(data)*0.7):]

print(f'Numero de textos de entrenamiento: {len(train_texts)}')
print(f'Numero de textos de prueba: {len(test_texts)}')

Numero de textos de entrenamiento: 6605
Numero de textos de prueba: 3129


#### 🔎 Resultado

6.605 frases de entrenamiento y 3.129 de prueba.

veamos algunos ejemplos:

### 📘 Primeras filas

**Objetivo de la celda.** Ver el formato de los datos antes de procesarlos.

**Qué hace cada parte del código**

| Parte del código | Qué hace |
|---|---|
| `train_texts.head()` | Muestra las cinco primeras filas de entrenamiento |

In [45]:
train_texts.head()

,ingles,emojis
0,the ratification and implementation of the upd...,✅🗳️
1,sustainable agriculture and biogas review of e...,🌱♻️🇪🇺
2,before the vote on paragraph,🗳️📄
3,plus we agreed to delete point a,❌
4,parliament agreed to accept the oral amendment,✅💬


#### 🔎 Resultado

Frases del Parlamento Europeo con su resumen en emojis: ✅ aprobar, ❌ eliminar, 🗳️ votar.

# **1. Preprocesamiento**
---
En este primer punto deberá implementar una función de prerpocesamiento de texto adaptada al problema presentado. Para eso deberá:


**Para la columna de texto:**

* Convertir todo el texto a minúsculas
* Eliminar caracteres especiales (mantener solo letras y espacios)
* Eliminar espacios repetidos
* Añadir los tokens de inicio `[BOS]` y final `[EOS]`

**Para la columna de emojis:**

* Añadir los tokens de inicio `[BOS]` y final `[EOS]`

**Entradas**:

* **`data`**: `pd.DataFrame`,  DataFrame que contiene las columnas 'texto' y 'emojis'

**Salida**:

* **`text_out`**: `pd.DataFrame`, DataFrame con el texto preprocesado según las reglas anteriores

### 📘 Punto 1 · preprocess

**Objetivo de la celda.** Limpiar el texto en inglés y marcar el inicio y el fin de cada secuencia en ambas columnas.

**Qué hace cada parte del código**

| Parte del código | Qué hace |
|---|---|
| `text.lower()` | Pasa a minúsculas: «The» y «the» serán el mismo token |
| `re.sub(r'[^a-z\s]', '', text)` | Borra todo lo que no sea letra minúscula o espacio: cifras, puntuación y apóstrofes |
| `re.sub(r'\s+', ' ', text).strip()` | Deja un solo espacio entre palabras y quita los de los extremos |
| `'[BOS] ' + text + ' [EOS]'` | Añade los tokens de inicio y fin, separados por espacios |
| `preprocess_emoji` | Solo añade `[BOS]` y `[EOS]`; los emojis no se tocan |
| `data.copy()` | Trabaja sobre una copia para no modificar el original |
| `.map(preprocess_text)` | Aplica la limpieza a cada celda de la columna |

> **Punto clave.** Los emojis van **sin separar**. La salida esperada del enunciado los muestra separados, y la primera versión los separaba: obtuvo 80. La instrucción escrita dice solo «añadir los tokens», y esa es la versión que da 100.

In [46]:
# FUNCIÓN CALIFICADA preprocess
def preprocess(data):
    def preprocess_text(text):
        text = text.lower()                         # minúsculas
        text = re.sub(r'[^a-z\s]', '', text)        # solo letras y espacios
        text = re.sub(r'\s+', ' ', text).strip()    # espacios repetidos
        return '[BOS] ' + text + ' [EOS]'
    def preprocess_emoji(emoji):
        return '[BOS] ' + emoji + ' [EOS]'

    text_out = data.copy()
    text_out['ingles'] = text_out['ingles'].map(preprocess_text)
    text_out['emojis'] = text_out['emojis'].map(preprocess_emoji)
    return text_out

### 📘 Prueba local del punto 1

**Objetivo de la celda.** Aplicar `preprocess` a ambos conjuntos y revisar la fila 0.

**Qué hace cada parte del código**

| Parte del código | Qué hace |
|---|---|
| `preprocess(train_texts.copy())` | Preprocesa el entrenamiento |
| `preprocess(test_texts.copy())` | Preprocesa la prueba |
| `iloc[0]['ingles'], iloc[0]['emojis']` | Toma la primera fila de cada columna para imprimirla |

In [47]:
# TEST CELL
train_texts_preprocess = preprocess(train_texts.copy())
test_texts_preprocess = preprocess(test_texts.copy())


print('Texto en inglés')
print(train_texts_preprocess.iloc[0]['ingles'])
print('Emojis')
print(train_texts_preprocess.iloc[0]['emojis'])

Texto en inglés
[BOS] the ratification and implementation of the updated ilo conventions vote [EOS]
Emojis
[BOS] ✅🗳️ [EOS]


#### 🔎 Resultado

La frase queda en minúsculas y delimitada, y los emojis conservan su forma original, **sin separar**.

**Salida esperada:**
```
Texto en ingles
[BOS] the ratification and implementation of the updated ilo conventions vote [EOS]
Emojis
[BOS] ✅ 🗳 ️ [EOS]
```




## **Evaluar código**

### 📘 Pruebas del punto 1

**Objetivo de la celda.** Verificar con el calificador el resultado de `preprocess` sobre sus propios textos. Esta explicación cubre las dos celdas siguientes, `Test 1_1` y `Test 1_2`.

> **Punto clave.** El calificador usa frases con mayúsculas, puntuación y paréntesis, y compara el DataFrame completo.

In [48]:
grader.run_test("Test 1_1", globals())

Test 1_1


In [49]:
grader.run_test("Test 1_2", globals())

Test 1_2


# **2. Vectorización**
----
Cree y ajuste una capa de vectorización para cada tipo de secuencia (texto y emojis).

**Entrada**:
* **`X`**:  `pd.DataFrame`, DataFrame que contiene el texto normalizado en `ingles` y `emoji`.
* **`seq_len`**: Tamaño máximo de la secuencia.
* **`vocab_size`**: Tamaño máximo del vocabulario.

**Salida**:
* **`en_vectorizer`**:  `keras.layer`, Vectorizador ajustado para el idioma inglés.

* **`em_vectorizer`**:  `keras.layer`, Vectorizador ajustado para el idioma emojis.


> **Nota**:Tenga presente que en el caso de la columna emojis su función solo debera añadir los tokens de incio y de parada, mientras que el texto debe ser preprocesado.

### 📘 Punto 2 · vectorize

**Objetivo de la celda.** Construir un vocabulario para el inglés y otro para los emojis, y convertir cada texto en una fila de enteros de longitud fija.

**Qué hace cada parte del código**

| Parte del código | Qué hace |
|---|---|
| `TextVectorization(max_tokens=vocab_size)` | Crea la capa con un vocabulario máximo de 5.000 tokens; el resto se vuelve `[UNK]` |
| `output_mode='int'` | Convierte cada token en su id entero |
| `output_sequence_length=seq_len + 1` | Rellena o recorta a 26; el +1 es por el desfase del entrenamiento |
| `standardize=None` | Desactiva la limpieza por defecto, que borraría los corchetes de `[BOS]` y `[EOS]` |
| `en_vectorizer.adapt(X['ingles'].values)` | Construye el vocabulario del inglés con el entrenamiento |
| `em_vectorizer.adapt(X['emojis'].values)` | Construye el vocabulario de los emojis con el entrenamiento |

> **Punto clave.** Los cuatro primeros ids son fijos: 0 relleno, 1 `[UNK]`, 2 `[EOS]` y 3 `[BOS]`. Por eso la generación del final arranca en 3 y se detiene en 2.

In [50]:
# FUNCIÓN CALIFICADA vectorize
def vectorize(X, seq_len, vocab_size):
    en_vectorizer = keras.layers.TextVectorization(
        max_tokens=vocab_size, output_mode='int',
        output_sequence_length=seq_len + 1, standardize=None)
    em_vectorizer = keras.layers.TextVectorization(
        max_tokens=vocab_size, output_mode='int',
        output_sequence_length=seq_len + 1, standardize=None)
    en_vectorizer.adapt(X['ingles'].values)
    em_vectorizer.adapt(X['emojis'].values)
    return en_vectorizer, em_vectorizer

### 📘 Prueba local del punto 2

**Objetivo de la celda.** Crear los vectorizadores y transformar los cuatro conjuntos de datos.

**Qué hace cada parte del código**

| Parte del código | Qué hace |
|---|---|
| `vectorize(train_texts_preprocess, 25, 5000)` | Construye ambos vectorizadores con el entrenamiento |
| `en_vectorizer(...).numpy()` | Convierte el inglés en una matriz de enteros |
| `emoji_vectorizer(...).numpy()` | Convierte los emojis en una matriz de enteros |

In [51]:
# TEST_CELL
en_vectorizer, emoji_vectorizer = vectorize(train_texts_preprocess, 25, 5000)
train_en_vec, train_emoji_vec = en_vectorizer(train_texts_preprocess['ingles']).numpy(), emoji_vectorizer(train_texts_preprocess['emojis']).numpy()
test_en_vec, test_emoji_vec = en_vectorizer(test_texts_preprocess['ingles']).numpy(), emoji_vectorizer(test_texts_preprocess['emojis']).numpy()

print(f'Tamaño de las entradas: {train_en_vec.shape}')
print(f'Tamaño de las salidas: {train_emoji_vec.shape}')

Tamaño de las entradas: (6605, 26)
Tamaño de las salidas: (6605, 26)


#### 🔎 Resultado

Matrices de 6.605 × 26. Como los emojis no llevan espacios, **cada cadena completa es un solo token**: `🤔❓` y `❓🤔` son tokens distintos. Toda fila de emojis queda `[3, id, 2, 0, …]`, lo que hace la tarea más difícil.

**Salida esperada**:
```
Tamaño de las entradas: (6605, 26)
Tamaño de las salidas: (6605, 26)

```

## **Evaluar código**

### 📘 Pruebas del punto 2

**Objetivo de la celda.** Verificar con el calificador los vectorizadores. Esta explicación cubre las dos celdas siguientes, `Test 2_1` y `Test 2_2`.

> **Punto clave.** Las dos dieron 100 en el primer intento.

In [52]:
grader.run_test("Test 2_1", globals())

Test 2_1


In [53]:
grader.run_test("Test 2_2", globals())

Test 2_2


# **3. Codificador**
----
Implemente una capa personalizada de Keras que represente el codificador de un Transformer.


**Entrada**:
* **`vocab_size`**: `int`, Tamaño del vocabulario de entrada
* **`maxlen`**: `int`, Longitud máxima de la secuencia de entrada
* **`embed_dim`**: `int`, Dimensión del espacio de embedding para cada token
* **`num_heads`**: `int`, Número de cabezas en el mecanismo de auto-atención
* **`ff_dim`**: `int`, Dimensión de la capa oculta en la red feed-forward
* **`rate`**: `float`, Tasa de dropout para regularización

**Salida**:
* **`layer`**:  `keras.layer.layer`, Capa personalizada de _Keras_.

### 📘 Punto 3 · get_EncoderLayer

**Objetivo de la celda.** Definir el codificador: leer la frase en inglés y producir un vector por palabra que ya incorpora el contexto de toda la frase.

**Qué hace cada parte del código**

| Parte del código | Qué hace |
|---|---|
| `self.token_emb = Embedding(vocab_size, embed_dim)` | Un vector de 128 por palabra |
| `self.pos_emb = Embedding(maxlen + 1, embed_dim)` | Un vector por posición; 26 porque la entrada mide 26 |
| `self.self_att = MultiHeadAttention(..., dropout=rate)` | Auto-atención con 2 cabezas y dropout en los pesos de atención |
| `self.self_layernorm, self.self_dropout` | Normalización y dropout del bloque de atención |
| `self.ffn = Sequential([Dense(ff_dim, relu), Dense(embed_dim)])` | Red de dos capas por posición: 128 → 32 → 128 |
| `self.dropout, self.layernorm` | Dropout y normalización del bloque feed-forward |
| `keras.ops.arange(0, longitud)` | Genera las posiciones de la secuencia |
| `token_emb(inputs) + pos_emb(positions)` | Representa cada token por qué palabra es y dónde está |
| `self_att(x, x)` | Cada posición mezcla información de las 26 |
| `self_layernorm(x + attn)` | Suma residual y normalización |
| `layernorm(x + f)` | Segundo subbloque con la misma estructura; devuelve el contexto |

> **Punto clave.** Los nombres de los atributos son parte de la respuesta: la prueba los inspecciona uno por uno. La primera versión usaba nombres propios y obtuvo 0; también hacía falta `dropout=rate` en la atención.

In [54]:
# FUNCIÓN CALIFICADA get_EncoderLayer
def get_EncoderLayer(vocab_size, maxlen, embed_dim, num_heads, ff_dim, rate):

    class encoder(keras.layers.Layer):
        def __init__(self, vocab_size, maxlen, embed_dim, num_heads, ff_dim, rate=0.1):
            super().__init__()
            self.token_emb = keras.layers.Embedding(input_dim=vocab_size, output_dim=embed_dim)
            self.pos_emb = keras.layers.Embedding(input_dim=maxlen + 1, output_dim=embed_dim)
            self.self_att = keras.layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim, dropout=rate)
            self.self_layernorm = keras.layers.LayerNormalization()
            self.self_dropout = keras.layers.Dropout(rate)
            self.ffn = keras.Sequential([keras.layers.Dense(ff_dim, activation="relu"),
                                         keras.layers.Dense(embed_dim)])
            self.dropout = keras.layers.Dropout(rate)
            self.layernorm = keras.layers.LayerNormalization()

        def call(self, inputs):
            positions = keras.ops.arange(start=0, stop=keras.ops.shape(inputs)[-1], step=1)
            x = self.token_emb(inputs) + self.pos_emb(positions)
            attn = self.self_att(x, x)
            attn = self.self_dropout(attn)
            x = self.self_layernorm(x + attn)
            f = self.ffn(x)
            f = self.dropout(f)
            return self.layernorm(x + f)

    layer = encoder(vocab_size, maxlen, embed_dim, num_heads, ff_dim, rate)
    return layer

### 📘 Prueba local del punto 3

**Objetivo de la celda.** Codificar 10 frases y comprobar la forma de la salida.

**Qué hace cada parte del código**

| Parte del código | Qué hace |
|---|---|
| `get_EncoderLayer(5000, 25, 128, 2, 32, .1)` | Crea el codificador con los hiperparámetros de la prueba |
| `encoder_layer(train_en_vec[:10])` | Codifica las 10 primeras frases |

In [55]:
# TEST_CELL
encoder_layer = get_EncoderLayer(5000, 25, 128, 2, 32, .1)
encoded_x = encoder_layer(train_en_vec[:10])
print(f'Codificación de sequencias de entrenamiento: {encoded_x.shape}')

Codificación de sequencias de entrenamiento: (10, 26, 128)


#### 🔎 Resultado

`(10, 26, 128)`: 26 vectores de 128 por frase, cada uno con información de toda la frase.

**Salida esperada**
```
Codificación de sequencias de entrenamiento: (10, 26, 128)
```

## **Evaluar código**

### 📘 Pruebas del punto 3

**Objetivo de la celda.** Verificar con el calificador el codificador. Esta explicación cubre las dos celdas siguientes.

**Qué hace cada parte del código**

| Parte del código | Qué hace |
|---|---|
| `grader.run_test("Test 3_1", ...)` | Revisa la estructura: nombres, tipos y parámetros de cada atributo |
| `grader.run_test("Test 3_2", ...)` | Compara los valores numéricos de la salida con la referencia |

In [56]:
grader.run_test("Test 3_1", globals())

Test 3_1


In [57]:
grader.run_test("Test 3_2", globals())

Test 3_2


# **4. Decodificador**
----
Implemente una capa personalizada de Keras que represente el decodificador de un Transformer.

**Entrada**:

* **`vocab_size`**: `int`, Tamaño del vocabulario de salida
* **`maxlen`**: `int`, Longitud máxima de la secuencia de salida  
* **`embed_dim`**: `int`, Dimensión del espacio de embedding para cada token
* **`num_heads`**: `int`, Número de cabezas en los mecanismos de atención
* **`ff_dim`**: `int`, Dimensión de la capa oculta en la red feed-forward
* **`rate`**: `float`, Tasa de dropout para regularización

**Salida**:
* **`layer`**:  `keras.layer.layer`, Capa personalizada de _Keras_.

### 📘 Punto 4 · get_DecoderLayer

**Objetivo de la celda.** Definir el decodificador: escribir los emojis mirando solo lo ya escrito y consultando la frase en inglés.

**Qué hace cada parte del código**

| Parte del código | Qué hace |
|---|---|
| `self.pos_emb = Embedding(maxlen, embed_dim)` | Un vector por posición; 25 porque la entrada son los emojis sin su último token |
| `self.self_att, self_layernorm, self_dropout` | Auto-atención sobre los emojis, con su normalización y dropout |
| `self.cross_att, cross_layernorm, cross_dropout` | Atención cruzada hacia el codificador, con su normalización y dropout |
| `self.ffn, dropout, layernorm` | Red feed-forward y su normalización |
| `self.out_dropout, self.out_layer = Dense(vocab_size, softmax)` | Dropout final y una probabilidad por cada token de emojis |
| `self_att(x, x, use_causal_mask=True)` | Máscara causal: cada posición solo mira las anteriores |
| `cross_att(x, context_tokens)` | Consultas desde los emojis; claves y valores desde el inglés |
| `return self.out_layer(x)` | Devuelve las probabilidades de cada posición |

> **Punto clave.** Sin la máscara causal, el modelo vería al entrenar el token que debe predecir y al generar fallaría. La atención cruzada es la única vía por la que el inglés influye en la salida.

In [58]:
# FUNCIÓN CALIFICADA get_DecoderLayer
def get_DecoderLayer(vocab_size, maxlen, embed_dim, num_heads, ff_dim, rate):

    class Decoder(keras.layers.Layer):
        def __init__(self, vocab_size, maxlen, embed_dim, num_heads, ff_dim, rate=0.1):
            super().__init__()
            self.token_emb = keras.layers.Embedding(input_dim=vocab_size, output_dim=embed_dim)
            self.pos_emb = keras.layers.Embedding(input_dim=maxlen, output_dim=embed_dim)
            self.self_att = keras.layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim, dropout=rate)
            self.self_layernorm = keras.layers.LayerNormalization()
            self.self_dropout = keras.layers.Dropout(rate)
            self.cross_att = keras.layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim, dropout=rate)
            self.cross_layernorm = keras.layers.LayerNormalization()
            self.cross_dropout = keras.layers.Dropout(rate)
            self.ffn = keras.Sequential([keras.layers.Dense(ff_dim, activation="relu"),
                                         keras.layers.Dense(embed_dim)])
            self.dropout = keras.layers.Dropout(rate)
            self.layernorm = keras.layers.LayerNormalization()
            self.out_dropout = keras.layers.Dropout(rate)
            self.out_layer = keras.layers.Dense(vocab_size, activation="softmax")

        def call(self, inputs, context_tokens):
            positions = keras.ops.arange(start=0, stop=keras.ops.shape(inputs)[-1], step=1)
            x = self.token_emb(inputs) + self.pos_emb(positions)
            attn = self.self_att(x, x, use_causal_mask=True)   # auto-atención causal
            attn = self.self_dropout(attn)
            x = self.self_layernorm(x + attn)
            attn = self.cross_att(x, context_tokens)           # atención cruzada con el encoder
            attn = self.cross_dropout(attn)
            x = self.cross_layernorm(x + attn)
            f = self.ffn(x)
            f = self.dropout(f)
            x = self.layernorm(x + f)
            x = self.out_dropout(x)
            return self.out_layer(x)

    layer = Decoder(vocab_size, maxlen, embed_dim, num_heads, ff_dim, rate)
    return layer

### 📘 Prueba local del punto 4

**Objetivo de la celda.** Decodificar 10 secuencias de emojis usando el contexto del codificador.

**Qué hace cada parte del código**

| Parte del código | Qué hace |
|---|---|
| `train_emoji_vec[:10, :-1]` | Emojis de 10 frases sin su último token: 25 posiciones |
| `decoder_layer(..., encoded_x)` | Les pasa también la salida del codificador como contexto |

In [59]:
# TEST_CELL
decoder_layer = get_DecoderLayer(5000, 25, 128, 2, 32, .1)
decoded_x = decoder_layer(train_emoji_vec[:10,:-1],encoded_x)
print(f'Decodificación de sequencias de entrenamiento: {decoded_x.shape}')

Decodificación de sequencias de entrenamiento: (10, 25, 5000)


#### 🔎 Resultado

`(10, 25, 5000)`: para cada frase y cada posición, una distribución de probabilidad sobre los 5.000 tokens.

## **Evaluar código**

### 📘 Pruebas del punto 4

**Objetivo de la celda.** Verificar con el calificador el decodificador, en estructura y en valores. Esta explicación cubre las dos celdas siguientes, `Test 4_1` y `Test 4_2`.

> **Punto clave.** Necesitó las mismas correcciones que el codificador: nombres exactos de los atributos y `dropout=rate` en las dos atenciones.

In [60]:
grader.run_test("Test 4_1", globals())

Test 4_1


In [61]:
grader.run_test("Test 4_2", globals())

Test 4_2


# **5. Construir modelo**
----
Implemente la función `get_model` que construye el modelo Transformer completo combinando el codificador y decodificador. El modelo debe realizar la tarea de secuencia-a-secuencia para convertir texto en inglés a representaciones con emojis.

**Entrada**:

* **`max_len`**: `int`, Longitud máxima de las secuencias
* **`vocab_size`**: `int`, Tamaño del vocabulario (debe ser el mismo para ambos vocabularios)
* **`embed_dim`**: `int`, Dimensión del espacio de embedding para cada token
* **`num_heads`**: `int`, Número de cabezas de atención
* **`ff_dim`**: `int`, Dimensión de la capa feed-forward
* **`rate`**: `float`, Tasa de dropout para regularización

**Salida**:
* **`model`**: `keras.Model`, Modelo Transformer encoder-decoder completo

> **Nota**: El modelo implementa la arquitectura encoder-decoder típica para tareas de traducción/generación de secuencias, donde el decodificador utiliza tanto su entrada desplazada como el contexto del codificador.

### 📘 Punto 5 · get_model

**Objetivo de la celda.** Unir codificador y decodificador en un modelo con dos entradas.

**Qué hace cada parte del código**

| Parte del código | Qué hace |
|---|---|
| `semillas y clear_session()` | Venían dadas: el modelo sale idéntico en cada llamada |
| `keras.Input(shape=(max_len + 1,))` | Entrada del inglés: 26 posiciones |
| `keras.Input(shape=(max_len,))` | Entrada de los emojis desplazados: 25 posiciones |
| `context = enc(encoder_inputs)` | El inglés pasa por el codificador |
| `outputs = dec(decoder_inputs, context)` | Los emojis y el contexto pasan por el decodificador |
| `keras.Model([encoder_inputs, decoder_inputs], outputs)` | Registra el grafo; el orden de la lista es el que espera `fit` |

In [62]:
# FUNCIÓN CALIFICADA get_model
def get_model(max_len, vocab_size, embed_dim, num_heads, ff_dim, rate):
    # Fijamos una semilla para efectos de reproducibiidad
    np.random.seed(0)
    keras.utils.set_random_seed(0)

    keras.backend.clear_session()

    encoder_inputs = keras.Input(shape=(max_len + 1,))
    decoder_inputs = keras.Input(shape=(max_len,))
    enc = get_EncoderLayer(vocab_size, max_len, embed_dim, num_heads, ff_dim, rate)
    dec = get_DecoderLayer(vocab_size, max_len, embed_dim, num_heads, ff_dim, rate)
    context = enc(encoder_inputs)
    outputs = dec(decoder_inputs, context)
    model = keras.Model([encoder_inputs, decoder_inputs], outputs)
    return model

### 📘 Prueba local del punto 5

**Objetivo de la celda.** Construir el modelo con los hiperparámetros de la prueba y comparar su resumen con el esperado.

**Qué hace cada parte del código**

| Parte del código | Qué hace |
|---|---|
| `max_len = 25, vocab_size = 5000` | Longitud y vocabulario |
| `embed_dim = 128, num_heads = 2, ff_dim = 32` | Tamaño de vectores, cabezas y red interna |
| `model.summary()` | Muestra capas, formas y número de parámetros |

In [63]:
# TEST_CELL
max_len = 25
vocab_size = 5000
embed_dim = 128
num_heads = 2
ff_dim = 32
rate = 0.01

model = get_model(max_len, vocab_size, embed_dim, num_heads, ff_dim, rate)

model.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 26)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, 25)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder (encoder)   │ (None, 26, 128)   │    784,160 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder (Decoder)   │ (None, 25, 5000)  │  1,561,256 │ input_layer_1[0]… │
│                     │                   │            │ encoder[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 2,345,416 (8.95 MB)

 Trainable params: 2,345,416 (8.95 MB)

 Non-trainable params: 0 (0.00 B)

#### 🔎 Resultado

Coincide con la salida esperada: **2.345.416 parámetros**.

| Componente | Codificador | Decodificador |
|---|---:|---:|
| Embedding de tokens | 640.000 | 640.000 |
| Embedding de posiciones | 3.328 | 3.200 |
| Atención | 131.968 | 263.936, son dos |
| Feed-forward | 8.352 | 8.352 |
| Normalizaciones | 512 | 768 |
| Capa de salida | — | 645.000 |
| **Total** | **784.160** | **1.561.256** |

**Salida esperada:**
```
Model: "functional_2"
┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 26)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, 25)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder (encoder)   │ (None, 26, 128)   │    784,160 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder (Decoder)   │ (None, 25, 5000)  │  1,561,256 │ input_layer_1[0]… │
│                     │                   │            │ encoder[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘
 Total params: 2,345,416 (8.95 MB)
 Trainable params: 2,345,416 (8.95 MB)
 Non-trainable params: 0 (0.00 B)
```

## **Evaluar código**

### 📘 Pruebas del punto 5

**Objetivo de la celda.** Verificar con el calificador el modelo ensamblado. Esta explicación cubre las dos celdas siguientes, `Test 5_1` y `Test 5_2`.

> **Punto clave.** Las dos dieron 100 en el primer intento.

In [64]:
grader.run_test("Test 5_1", globals())

Test 5_1


In [65]:
grader.run_test("Test 5_2", globals())

Test 5_2


# **6. Entrenar el modelo**
----
Complete la función `train_model` que recibe un modelo y lo ajusta usando las secuencias de entrenamiento.

**Entrada**:

* **`model`**: `keras.Model`, modelo de _Keras_ compilado.
* **`train_text`**:  `array`, Arreglo con los textos vectorizados de entrenamiento en inglés.
* **`train_emoji`**:  `array`, Arreglo con los textos vectorizados de entrenamiento de emojis.
* **`test_text`**:  `array`, Arreglo con los textos vectorizados de prueba en inglés.
* **`test_emoji`**:  `array`, Arreglo con los textos vectorizados de prueba en emojis.
* **`learning_rate`**: `float`, tasa de aprendizaje.
* **`epochs`**: `int`, Número de epocas.
* **`batch_size`**: `int`, tamaño del lote.

**Salida**:
* **`history`**: Objeto, objeto que contiene las métricas de evaluación y entrenamiento a lo largo de las épocas.

> **Nota**: Recuerde desfasar por un token la salida esperada de la entrada.



### 📘 Función de pérdida causal_loss, dada por la plantilla

**Objetivo de la celda.** Medir el error de predicción ignorando las posiciones de relleno.

**Qué hace cada parte del código**

| Parte del código | Qué hace |
|---|---|
| `masc = cast(y_true != 0)` | Vale 1 en los tokens reales y 0 en el relleno |
| `one_hot(y_true, 5000)` | Convierte cada id en un vector de 5.000 con un 1 en su posición |
| `CategoricalCrossentropy(from_logits=False, reduction='none')` | Error por posición; espera probabilidades, no logits |
| `loss * masc` | Anula el error en el relleno |
| `sum(loss) / sum(masc)` | Promedia solo sobre los tokens reales |

> **Punto clave.** Cada fila tiene solo dos posiciones que cuentan: la cadena de emojis y el `[EOS]`.

In [66]:
def causal_loss(y_true,y_pred):
    masc = keras.ops.cast(y_true != 0, dtype="float32")
    y_true = keras.ops.one_hot(y_true, 5000, axis=-1)

    loss_fn = keras.losses.CategoricalCrossentropy(from_logits=False, reduction='none')
    loss = loss_fn(y_true, y_pred)
    loss = loss * masc
    loss = keras.ops.sum(loss)/keras.ops.sum(masc)
    return loss

### 📘 Punto 6 · train_model

**Objetivo de la celda.** Compilar el modelo y entrenarlo desfasando la salida un token respecto a la entrada.

**Qué hace cada parte del código**

| Parte del código | Qué hace |
|---|---|
| `model.compile(optimizer=Adam(...), loss=causal_loss, metrics=['accuracy'])` | Fija optimizador, pérdida y métrica; `get_model` no compila |
| `train_emoji[:, :-1]` | Entrada del decodificador: los emojis sin el último token |
| `train_emoji[:, 1:]` | Objetivo: los emojis sin el primero |
| `validation_data=(...)` | Aplica el mismo desfase a la prueba |
| `epochs, batch_size` | Número de épocas y tamaño de lote recibidos |

> **Punto clave.** Es el *teacher forcing*: en la posición 0 el modelo ve `[BOS]` y debe decir el primer emoji; en la 1 ve ese emoji y debe decir `[EOS]`.

In [67]:
# FUNCIÓN CALIFICADA train_model
def train_model (model, train_text, train_emoji, test_text, test_emoji, learning_rate, epochs, batch_size):
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
                  loss=causal_loss, metrics=['accuracy'])
    # El decodificador recibe la secuencia sin el último token
    # y debe predecir la secuencia desfasada un token
    history = model.fit(
        [train_text, train_emoji[:, :-1]], train_emoji[:, 1:],
        validation_data=([test_text, test_emoji[:, :-1]], test_emoji[:, 1:]),
        epochs=epochs, batch_size=batch_size)

    return history

### 📘 Entrenamiento de 40 épocas

**Objetivo de la celda.** Entrenar el modelo con la configuración del enunciado.

**Qué hace cada parte del código**

| Parte del código | Qué hace |
|---|---|
| `train_model(model, ..., 1e-4, 40, 256)` | Tasa de aprendizaje 0,0001, 40 épocas y lotes de 256: 26 pasos por época |

In [68]:
history = train_model(model, train_en_vec, train_emoji_vec, test_en_vec, test_emoji_vec, 1e-4, 40, 256)

Epoch 1/40
26/26 ━━━━━━━━━━━━━━━━━━━━ 38s 755ms/step - accuracy: 0.0256 - loss: 8.0254 - val_accuracy: 0.0400 - val_loss: 7.2922
Epoch 2/40
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 41ms/step - accuracy: 0.0400 - loss: 7.2663 - val_accuracy: 0.0400 - val_loss: 6.6983
Epoch 3/40
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 38ms/step - accuracy: 0.0400 - loss: 6.8950 - val_accuracy: 0.0400 - val_loss: 6.2723
Epoch 4/40
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 38ms/step - accuracy: 0.0400 - loss: 6.6223 - val_accuracy: 0.0400 - val_loss: 5.8834
Epoch 5/40
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 38ms/step - accuracy: 0.0400 - loss: 6.3719 - val_accuracy: 0.0400 - val_loss: 5.5077
Epoch 6/40
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 38ms/step - accuracy: 0.0400 - loss: 6.1313 - val_accuracy: 0.0400 - val_loss: 5.1421
Epoch 7/40
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 45ms/step - accuracy: 0.0400 - loss: 5.8985 - val_accuracy: 0.0400 - val_loss: 4.7895
Epoch 8/40
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 41ms/step - accuracy: 0.0400 - loss: 5.6773 - val_accuracy: 0.0400 -

#### 🔎 Resultado

| Época | Pérdida | Pérdida de validación | Accuracy de validación |
|---:|---:|---:|---:|
| 1 | 8,03 | 7,29 | 0,040 |
| **19** | 4,30 | **3,11** | 0,053 |
| 40 | 3,48 | 3,85 | 0,047 |

- **Aprendió:** la pérdida empezó cerca de ln 5.000 ≈ 8,5, que equivale a adivinar al azar, y bajó a 3,48.
- **Sobreajuste desde la época 19:** la validación tocó su mínimo y luego subió mientras el entrenamiento seguía bajando.
- **Accuracy de 0,04 = 1/25:** la métrica no tiene máscara y cuenta las 25 posiciones; el modelo acierta casi solo el `[EOS]`. Aquí la métrica útil es la pérdida de validación.

**Salida esperada**
```
Epoch 39/40
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 39ms/step - accuracy: 0.0804 - loss: 0.1294 - val_accuracy: 0.0411 - val_loss: 4.8453
Epoch 40/40
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 39ms/step - accuracy: 0.0805 - loss: 0.1232 - val_accuracy: 0.0409 - val_loss: 4.8631
```

veamos algunos ejemplos:

### 📘 Generación de un ejemplo, dada por la plantilla

**Objetivo de la celda.** Traducir a emojis una frase de prueba que el modelo nunca vio, generando un token a la vez.

**Qué hace cada parte del código**

| Parte del código | Qué hace |
|---|---|
| `current_token = 3` | Arranca con `[BOS]` |
| `text_encoding = np.zeros([1, 25])` | Secuencia de emojis vacía, que se irá llenando |
| `model([frase, text_encoding])` | Predice probabilidades para las 25 posiciones |
| `np.argmax(token_prediction[0, i])` | Elige el token más probable en la posición actual |
| `text_encoding[0, i + 1] = ...` | Lo escribe en la siguiente posición |
| `if current_token == 2: break` | Se detiene al generar `[EOS]` |
| `get_vocabulary()` | Traduce los ids de vuelta a palabras y emojis |

In [69]:
current_token = 3
text_encoding = np.zeros([1,25])
text_encoding[0,0] = current_token

for i in range(24):
    token_prediction = model([np.expand_dims(test_en_vec[8], axis=0), text_encoding])
    current_token = np.argmax(token_prediction[0,i])
    text_encoding[0,i+1] = int(current_token)
    if current_token == 2:
        break

emoji_vocabulary = np.array(emoji_vectorizer.get_vocabulary())
en_vocabulary = np.array(en_vectorizer.get_vocabulary())
print(' '.join(en_vocabulary[list(test_en_vec[8].astype(int))]))
print(' '.join(emoji_vocabulary[list(text_encoding[0].astype(int))]))

[BOS] we also have a common responsibility to ensure that this takes place with legal certainty while [UNK] fundamental rights [EOS]     
[BOS] 🇪🇺❌🚫 [EOS]                      


#### 🔎 Resultado

Generó **🇪🇺❌🚫** y luego `[EOS]`. En lo mecánico funcionó; en el significado es parcial: 🇪🇺 encaja con el dominio, pero ❌🚫 no refleja «derechos fundamentales». Es coherente con una pérdida de validación cercana a 3,8.

## **Evaluar código**

### 📘 Prueba del punto 6

**Objetivo de la celda.** Verificar con el calificador la función `train_model`.

> **Punto clave.** Califica la función, no los números del registro, que dependen del hardware y de la versión de Keras.

In [70]:
grader.run_test("Test 6_1", globals())

Test 6_1
26/26 ━━━━━━━━━━━━━━━━━━━━ 33s 615ms/step - accuracy: 0.0256 - loss: 8.0254 - val_accuracy: 0.0400 - val_loss: 7.2922
26/26 ━━━━━━━━━━━━━━━━━━━━ 33s 631ms/step - accuracy: 0.0256 - loss: 8.0254 - val_accuracy: 0.0400 - val_loss: 7.2922


# **Evaluación**

### 📘 Envío oficial

**Objetivo de la celda.** Registrar la entrega en UNCode.

**Qué hace cada parte del código**

| Parte del código | Qué hace |
|---|---|
| `grader.submit_task(globals())` | Vuelve a ejecutar las 11 pruebas en el servidor y guarda la calificación |

In [71]:
grader.submit_task(globals())

Test 1_1
Test 1_2
Test 2_1
Test 2_2
Test 3_1
Test 3_2
Test 4_1
Test 4_2
Test 5_1
Test 5_2
Test 6_1
26/26 ━━━━━━━━━━━━━━━━━━━━ 33s 622ms/step - accuracy: 0.0256 - loss: 8.0254 - val_accuracy: 0.0400 - val_loss: 7.2922
26/26 ━━━━━━━━━━━━━━━━━━━━ 32s 618ms/step - accuracy: 0.0256 - loss: 8.0254 - val_accuracy: 0.0400 - val_loss: 7.2922


#### 🔎 Resultado

**Las 11 pruebas en 100 y «Your submission grade: 100.0%».**

Revisa en UNCode tus submissions en la [página de la tarea.](https://uncode.unal.edu.co/course/DL-GroupMLDS-5-2026-2/8c487bd7-4c95-4b75-88d5-9e7093aae729)

# 📘 Cierre

| Punto | Función | Primera versión | Final | Qué fue decisivo |
|---|---|---|---|---|
| 1 | `preprocess` | 80 | 100 | Emojis sin separar |
| 2 | `vectorize` | 100 | 100 | `standardize=None` y longitud 26 |
| 3 | `get_EncoderLayer` | 0 y 10 | 100 | Nombres exactos de atributos y `dropout=rate` |
| 4 | `get_DecoderLayer` | 0 y 100 | 100 | Lo mismo, más máscara causal y atención cruzada |
| 5 | `get_model` | 100 | 100 | Dos entradas, de 26 y 25 |
| 6 | `train_model` | 100 | 100 | Entrada `[:, :-1]` y objetivo `[:, 1:]` |

**Ideas para recordar**
1. El codificador lee y el decodificador escribe; se comunican por la atención cruzada.
2. La máscara causal permite generar de izquierda a derecha.
3. La pérdida ignora el relleno; la accuracy sin máscara no, y por eso engaña.
4. Un calificador automático también evalúa nombres, tipos y parámetros.

# **Créditos**
---

* **Profesor:** [Fabio Augusto Gonzalez](https://dis.unal.edu.co/~fgonza/)
* **Asistentes docentes :**
  * [Juan Sebastián Malagón Torres](https://http://juselara.com/)
* **Coordinador de virtualización:**
    - [Edder Hernández Forero](https://www.linkedin.com/in/edder-hernandez-forero-28aa8b207/).

**Universidad Nacional de Colombia** - *Facultad de Ingeniería*